In [1]:
from wizard import Node, Flow, InputField, RadioField, Next

In [2]:
profile_screen = Node(
    "profile_screen",
    "Profile",
    instructions="""Greet the user (If you have not greeted them before), introduce yourself, then ask the user for their name and address.""",
    fields=[
        InputField("tenant_name", "Tenant Name", "The name of the calling user/tenant"),
        InputField("tenant_address", "Tenant Address", "The address of the user/tenant")
    ],
    next="location_screen"
)

location_screen = Node(
    "location_screen",
    "Location",
    instructions="""Learn whether the issue is inside their home or outside""",
    fields=[
        RadioField("location", "Location", "Where the issue is located", [
            "Inside home",
            "Outside home"
        ])
    ],
    next=lambda ctx: "inside_home_area_screen" if ctx["location"] == "Inside home" else "outside_home_area_screen"
)


inside_home_area_screen = Node(
    "inside_home_area_screen",
    "Inside Home Area",
    instructions="Ask the user about the area where the issue is",
    fields=[
        RadioField("area", "Area", "", [
            "Floors, Walls and Stairs",
            "Plumbing",
            "Doors, Locks and Windows",
            "Electrics",
            "Alarms & Door Entry",
            "Heating & Hot Water",
            "Empty Repair"
        ])
    ],
    next={
        "Floors, Walls and Stairs": "floors_walls_stairs_screen",
        "Plumbing": "plumbing_screen",
        # "Roof leaking": "roof_leaking_screen"
    }
)

floors_walls_stairs_screen = Node(
    "floors_walls_stairs_screen",
    "Area: Floors, walls and Stairs",
    "Ask the user about the precise area where the issue is",
    fields=[
        RadioField(
            "area_tier_2",
            "Which Area excatly has the problem?",
            "",
            [
                "Floors",
                "Ceilings",
                "Walls",
                "Stairs"
            ]
        )
    ],
    next={
        "Ceilings": "ceilings_issues_screen",
        "Floors": "floor_issues_screen"
    }
)

ceiling_issues_screen = Node(
    "ceilings_issues_screen",
    "Ceilings Issues",
    "Learn from the user which kindo of ceiling issue they area dealing with",
    fields=[
        RadioField("ceiling_issue", "Ceiling Issue", "", [
            "Ceiling is falling down",
            "Cracks in the ceiling",
            "Roof leaking"
    ])
    ],
    next={
        "Ceiling is falling down": "ceiling_is_falling_down_screen",
        "Cracks in the ceiling": "cracks_in_the_ceiling_screen",
        "Roof leaking": "roof_leaking_screen"
    }
)

cracks_in_the_ceiling_screen = Node(
    "cracks_in_the_ceiling_screen",
    "Cracks in the ceiling",
    "Learn whether the crack can fit a 1 euro coin or not",
    fields=[
        RadioField("crack_can_fit_coin", "Could you fit a one euro coin in the gap?", "", 
                   [
                       "yes",
                       "no"
                   ])
    ],
    next="exact_location_screen"
)

In [3]:
"".join([w.title() for w in "Could you fit a one euro coin in the gap?".split()])

'CouldYouFitAOneEuroCoinInTheGap?'

In [12]:
flow = Flow([profile_screen, 
             location_screen, 
             inside_home_area_screen, 
             ceiling_issues_screen, 
             floors_walls_stairs_screen, 
             ceiling_issues_screen, 
             cracks_in_the_ceiling_screen])

flow.play_actions([Next()])
flow.play_actions([flow.current_action_model().__args__[0](value="Inside home")])
flow.play_actions([Next()])
# flow.play_actions([flow.current_action_model().__args__[0](value="Floors, Walls and Stairs")])
# flow.play_actions([Next()])
# flow.play_actions([flow.current_action_model().__args__[0](value="Ceilings")])
# flow.play_actions([Next()])
# flow.play_actions([flow.current_action_model().__args__[0](value="Cracks in the ceiling")])
# flow.play_actions([Next()])




print(flow.render())



Current Path: Profile > Location > Inside Home Area

Node: Inside Home Area
Instructions: Ask the user about the area where the issue is
---

Area (Radio Field)
( ) Floors, Walls and Stairs
( ) Plumbing
( ) Doors, Locks and Windows
( ) Electrics
( ) Alarms & Door Entry
( ) Heating & Hot Water
( ) Empty Repair


In [5]:
flow.current_action_model()

typing.Union[wizard.FillTenantName, wizard.FillTenantAddress, wizard.Next]

In [ ]:
import os
from typing import Optional

import openai


SYS = """
You are a customer support AI agent operating within our agent scripting system. Your role is to assist customers with their inquiries, issues, and requests in a helpful, professional, and efficient manner.
"""

async def call_llm(flow: Flow, event_stream: list):
    client = openai.AsyncOpenAI(api_key=os.environ["OPENAI_API_KEY"])

    class AgentOutput:
        response: str
        aciton: Optional[list[flow.current_action_model()]]
    
    event_stream_str = ...
    user_msg = f"<event_stream>\n{event_stream_str}\n</event_stream\n\n<agent_script>\n{flow.render()}\n</agent_script>"
    res = await client.beta.chat.completions.parse(
                model="gpt-4.1",
                messages=[
                    {
                        "role": "system",
                        "content": SYS,
                    },
                    {
                        "role": "user",
                        "content": user_msg,
                    },
                ],
                response_format=AgentOutput,
            )
    message = res.choices[0].message
    agent_output = message.parsed
    print(agent_output)
    if agent_output.get("actions"):
        for actions in actions:
            # run these actions
            ...
    return agent_output


In [ ]:
flow = Flow([profile_screen, 
             location_screen, 
             inside_home_area_screen, 
             ceiling_issues_screen, 
             floors_walls_stairs_screen, 
             ceiling_issues_screen, 
             cracks_in_the_ceiling_screen])

typing.Union[str, float, int]